# Lesson 5 — Shear Force Diagram

**Course:** Python for Structural Engineers — Simply Supported Beam  
**Prerequisites:** Lessons 1–4  
**You will learn:**
- Compute shear force at any section by summing forces to the left
- Build an array of 500 shear values in one loop
- Plot the SFD with colour fills for positive and negative zones
- Use `import` to bring in the `beam_solver` module (your code, reused)

**Time estimate:** 60 minutes

---

## The Theory — What Is Shear Force?

At any cross-section of the beam, the **shear force** V is the algebraic sum of all vertical forces acting to the **left** of that section:

$$V(x) = \sum_{\text{forces at } x_i \leq x} F_i$$

Where forces include:
- Applied loads (point loads, UDLs) — negative if downward
- Support reactions R_A and R_B — positive if upward

**Key rules for the SFD shape:**
- At a **point load**, V jumps by the magnitude of that load
- Under a **UDL**, V changes linearly (constant slope = w)
- At the **free ends** of cantilevers, V = 0
- At a **reaction**, V jumps by +R (upward)

---

## 5.1 Computing V by Hand at a Few Points

In [ ]:
import numpy as np
import sys
sys.path.insert(0, "../shared")
from plot_helpers import draw_beam, draw_loads, draw_sfd

# Beam and loads (same as Lesson 4)
geometry = {"L_total": 10.0, "x_A": 2.0, "x_B": 8.0}
loads = [
    {"type": "point", "x": 5.0,  "magnitude": -20.0},
    {"type": "udl",   "x1": 2.0, "x2": 8.0, "w": -5.0},
]

# Compute reactions (from our function in Lesson 4)
def compute_reactions(geometry, loads):
    x_A, x_B = geometry["x_A"], geometry["x_B"]
    span = x_B - x_A
    total_F = total_M = 0.0
    for load in loads:
        if load["type"] == "point":
            P = load["magnitude"]; total_F += P; total_M += P * (load["x"] - x_A)
        elif load["type"] == "udl":
            P = load["w"] * (load["x2"] - load["x1"])
            total_F += P; total_M += P * ((load["x1"] + load["x2"]) / 2 - x_A)
    R_B = -total_M / span
    R_A = -total_F - R_B
    return R_A, R_B

R_A, R_B = compute_reactions(geometry, loads)
print(f"R_A = {R_A:+.2f} kN,  R_B = {R_B:+.2f} kN")

In [ ]:
# Manually calculate V at a few positions
x_A, x_B = geometry["x_A"], geometry["x_B"]

test_positions = [0.0, 1.9, 2.1, 4.9, 5.1, 7.9, 8.1, 10.0]

print(f"{'x (m)':>8}   {'V (kN)':>10}   Notes")
print("-" * 55)

for xi in test_positions:
    v = 0.0
    notes = []

    if xi >= x_A:
        v += R_A
        notes.append(f"+R_A({R_A:+.1f})")
    if xi >= x_B:
        v += R_B
        notes.append(f"+R_B({R_B:+.1f})")

    for load in loads:
        if load["type"] == "point" and load["x"] <= xi:
            v += load["magnitude"]
            notes.append(f"P({load['magnitude']:+.0f})")
        elif load["type"] == "udl":
            overlap = min(xi, load["x2"]) - load["x1"]
            if overlap > 0:
                contrib = load["w"] * overlap
                v += contrib
                notes.append(f"UDL({contrib:+.1f})")

    print(f"{xi:>8.1f}   {v:>+10.2f}   {', '.join(notes)}")

Notice the **jumps** just before and after x_A, x_B, and the point load at 5 m. Those are the discontinuities in the SFD.

---

## 5.2 Automating the Calculation for 500 Points

In [ ]:
def compute_shear(geometry, loads, R_A, R_B, n_points=500):
    """Build the shear force array V(x) using the cut-section method."""
    L   = geometry["L_total"]
    x_A = geometry["x_A"]
    x_B = geometry["x_B"]

    x_array = np.linspace(0.0, L, n_points)   # 500 equally-spaced positions
    V       = np.zeros(n_points)               # start with all zeros

    for i, xi in enumerate(x_array):
        v = 0.0

        # Reactions (upward = positive)
        if xi >= x_A:
            v += R_A
        if xi >= x_B:
            v += R_B

        # Applied loads
        for load in loads:
            if load["type"] == "point":
                if load["x"] <= xi:
                    v += load["magnitude"]
            elif load["type"] == "udl":
                x1, x2 = load["x1"], load["x2"]
                overlap = min(xi, x2) - x1
                if overlap > 0:
                    v += load["w"] * overlap

        V[i] = v

    return x_array, V


x_array, V = compute_shear(geometry, loads, R_A, R_B)
print(f"V at x=0    : {V[0]:+.2f} kN  (should be 0 — free left end)")
print(f"V at x=10 m : {V[-1]:+.2f} kN  (should be ≈ 0 — free right end)")
print(f"V_max       : {V.max():+.2f} kN  at x = {x_array[V.argmax()]:.2f} m")
print(f"V_min       : {V.min():+.2f} kN  at x = {x_array[V.argmin()]:.2f} m")

---

## 5.3 Plotting the SFD

In [ ]:
import matplotlib.pyplot as plt

fig, (ax_beam, ax_sfd) = plt.subplots(2, 1, figsize=(13, 7),
                                       gridspec_kw={"height_ratios": [1, 2]})
fig.subplots_adjust(hspace=0.05)

# ── Top panel: beam diagram ───────────────────────────────────────────────
draw_beam(ax_beam, geometry)
draw_loads(ax_beam, loads, geometry["L_total"])

# ── Bottom panel: SFD ─────────────────────────────────────────────────────
ax_sfd.plot(x_array, V, color="#1565C0", linewidth=2.2, label="V(x)")
ax_sfd.fill_between(x_array, V, 0,
                    where=(V >= 0), color="#90CAF9", alpha=0.5, label="Positive (upward)")
ax_sfd.fill_between(x_array, V, 0,
                    where=(V < 0),  color="#EF9A9A", alpha=0.5, label="Negative (downward)")
ax_sfd.axhline(0, color="black", linewidth=0.8)

# Mark where V crosses zero (potential max moment location)
zero_crossings = np.where(np.diff(np.sign(V)))[0]
for idx in zero_crossings:
    x_zero = x_array[idx]
    ax_sfd.axvline(x_zero, color="red", linestyle="--", linewidth=0.9, alpha=0.7)
    ax_sfd.text(x_zero + 0.1, V.min() * 0.9,
                f"V=0\n({x_zero:.1f} m)", fontsize=7, color="red")

# Axis formatting
ax_sfd.set_xlim(ax_beam.get_xlim())
ax_sfd.set_ylabel("Shear Force V (kN)", fontsize=10)
ax_sfd.set_xlabel("Position x (m)", fontsize=10)
ax_sfd.legend(fontsize=8, loc="upper right")
ax_sfd.grid(True, linestyle="--", alpha=0.4)
ax_sfd.set_title("Shear Force Diagram", fontsize=11, pad=6)

plt.tight_layout()
plt.show()

---

## 5.4 Introducing `beam_solver.py` — Your Code as a Module

We have now written `compute_reactions` and `compute_shear` in two different notebooks. Every time we copy code, we risk having two versions that disagree. The solution: put the code in one file and **import** it.

The file `../shared/beam_solver.py` already contains both functions (plus `compute_moments` and `analyse` for Lesson 6). Let's use it:

In [ ]:
import sys
sys.path.insert(0, "../shared")
import beam_solver

# Exact same call, but now using the module
R_A, R_B = beam_solver.compute_reactions(geometry, loads)
x, V     = beam_solver.compute_shear(geometry, loads, R_A, R_B)

print(f"R_A = {R_A:+.2f} kN,  R_B = {R_B:+.2f} kN")
print(f"V[-1] = {V[-1]:+.4f} kN  (≈ 0 means equilibrium is correct)")

From now on, **we import rather than copy**. This is the professional way to write Python.

---

## 5.5 Interactive SFD Widget

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.insert(0, "../shared")
import beam_solver
from plot_helpers import draw_beam, draw_loads, draw_reactions, draw_sfd

%matplotlib inline

live_loads = []
out_plot   = widgets.Output()
out_status = widgets.Output()

style  = {"description_width": "170px"}
layout = widgets.Layout(width="460px")

# Geometry
sl_L  = widgets.FloatSlider(value=10.0, min=5.0, max=20.0, step=0.5,
                             description="Total length L (m):", style=style, layout=layout)
sl_xA = widgets.FloatSlider(value=2.0,  min=0.0, max=9.5,  step=0.5,
                             description="Support A at (m):",   style=style, layout=layout)
sl_xB = widgets.FloatSlider(value=8.0,  min=0.5, max=10.0, step=0.5,
                             description="Support B at (m):",   style=style, layout=layout)

# Load
dd_type = widgets.Dropdown(options=[("Point load", "point"), ("UDL", "udl")],
                            description="Load type:", style=style, layout=layout)
sl_x    = widgets.FloatSlider(value=5.0, min=0.0, max=10.0, step=0.5,
                               description="Position x (m):", style=style, layout=layout)
sl_P    = widgets.FloatSlider(value=-20.0, min=-150.0, max=150.0, step=5.0,
                               description="Magnitude (kN):", style=style, layout=layout)
sl_x1   = widgets.FloatSlider(value=2.0, min=0.0, max=9.5, step=0.5,
                               description="UDL start x1 (m):", style=style, layout=layout)
sl_x2   = widgets.FloatSlider(value=8.0, min=0.5, max=10.0, step=0.5,
                               description="UDL end x2 (m):",   style=style, layout=layout)
sl_w    = widgets.FloatSlider(value=-5.0, min=-40.0, max=40.0, step=1.0,
                               description="Intensity (kN/m):", style=style, layout=layout)
chk_zeros = widgets.Checkbox(value=True, description="Show V=0 sections")

btn_add   = widgets.Button(description="Add Load",  button_style="primary")
btn_clear = widgets.Button(description="Clear All", button_style="danger")
pt_ctrl   = widgets.VBox([sl_x, sl_P])
udl_ctrl  = widgets.VBox([sl_x1, sl_x2, sl_w])
udl_ctrl.layout.display = "none"

def on_type(c):
    if c["new"] == "point":
        pt_ctrl.layout.display = ""; udl_ctrl.layout.display = "none"
    else:
        pt_ctrl.layout.display = "none"; udl_ctrl.layout.display = ""
dd_type.observe(on_type, names="value")

def redraw(change=None):
    L   = sl_L.value
    x_A = min(sl_xA.value, sl_xB.value - 0.5)
    x_B = min(sl_xB.value, L)
    geo = {"L_total": L, "x_A": x_A, "x_B": x_B}

    with out_plot:
        out_plot.clear_output(wait=True)

        if not live_loads:
            fig, ax = plt.subplots(figsize=(13, 3.5))
            draw_beam(ax, geo)
            ax.set_title("Add loads to generate the SFD", fontsize=10, color="gray")
            plt.tight_layout(); plt.show()
            return

        try:
            R_A, R_B = beam_solver.compute_reactions(geo, live_loads)
            x, V = beam_solver.compute_shear(geo, live_loads, R_A, R_B)
        except Exception as e:
            print(f"Error: {e}"); return

        fig, (ax_beam, ax_sfd) = plt.subplots(
            2, 1, figsize=(13, 7.5),
            gridspec_kw={"height_ratios": [1, 2.2]}
        )
        fig.subplots_adjust(hspace=0.05)

        draw_beam(ax_beam, geo)
        draw_loads(ax_beam, live_loads, L)
        draw_reactions(ax_beam, x_A, x_B, R_A, R_B)

        ax_sfd.plot(x, V, color="#1565C0", lw=2.2)
        ax_sfd.fill_between(x, V, 0, where=(V>=0), color="#90CAF9", alpha=0.5)
        ax_sfd.fill_between(x, V, 0, where=(V<0),  color="#EF9A9A", alpha=0.5)
        ax_sfd.axhline(0, color="black", lw=0.8)

        if chk_zeros.value:
            zc = np.where(np.diff(np.sign(V)))[0]
            for idx in zc:
                ax_sfd.axvline(x[idx], color="red", ls="--", lw=1, alpha=0.7)
                ax_sfd.text(x[idx]+0.1, V.min()*0.8,
                            f"V=0\n({x[idx]:.1f}m)", fontsize=7, color="red")

        ax_sfd.set_xlim(ax_beam.get_xlim())
        ax_sfd.set_ylabel("Shear V (kN)", fontsize=10)
        ax_sfd.set_xlabel("Position x (m)", fontsize=10)
        ax_sfd.grid(True, ls="--", alpha=0.4)
        ax_sfd.set_title(
            f"SFD  |  R_A = {R_A:+.1f} kN  |  R_B = {R_B:+.1f} kN",
            fontsize=11, pad=6
        )
        plt.tight_layout(); plt.show()

for sl in [sl_L, sl_xA, sl_xB, chk_zeros]:
    sl.observe(redraw, names="value")
chk_zeros.observe(redraw, names="value")

def on_add(b):
    with out_status:
        out_status.clear_output(wait=True)
        if dd_type.value == "point":
            live_loads.append({"type": "point", "x": sl_x.value, "magnitude": sl_P.value})
            print(f"✓ {sl_P.value:+.0f} kN at x={sl_x.value:.1f} m")
        else:
            if sl_x2.value <= sl_x1.value:
                print("Error: x2 must be > x1"); return
            live_loads.append({"type": "udl", "x1": sl_x1.value,
                                "x2": sl_x2.value, "w": sl_w.value})
            print(f"✓ UDL {sl_w.value:+.0f} kN/m  [{sl_x1.value:.1f}–{sl_x2.value:.1f} m]")
    redraw()

def on_clear(b):
    live_loads.clear()
    with out_status:
        out_status.clear_output(wait=True); print("Cleared.")
    redraw()

btn_add.on_click(on_add)
btn_clear.on_click(on_clear)

redraw()
display(widgets.VBox([
    widgets.HTML("<b>Geometry</b>"),
    sl_L, sl_xA, sl_xB,
    widgets.HTML("<hr><b>Add Loads</b>"),
    dd_type, pt_ctrl, udl_ctrl,
    widgets.HBox([btn_add, btn_clear]),
    chk_zeros,
    out_status, out_plot
]))

**Experiments:**
1. Add a single midspan load → the SFD is a simple step function with V = R_A to the left of the load.
2. Add a UDL → watch the SFD slope linearly under the UDL region.
3. Add a load on the left cantilever → V at x=0 should be 0 (free end).
4. Toggle **Show V=0 sections** — the red dashed lines mark where the bending moment will be maximum.

---

## 5.6 Practice Exercise

Use `beam_solver.compute_shear()` to calculate V at exactly x = 4.0 m for:  
- Beam: L = 10 m, x_A = 2 m, x_B = 8 m  
- Loads: −15 kN at x = 3 m, −8 kN/m UDL from 0 to 10 m

Use `np.interp(4.0, x_array, V_array)` to pick the value at exactly x=4 from the array.

In [ ]:
# Your code here


---

## Summary

**Python concepts covered:**
- Building a value array with a `for` loop: `for i, xi in enumerate(x_array)`
- `np.zeros(n)` — creating an array of zeros
- `np.where(np.diff(np.sign(V)))` — finding zero-crossing indices
- `import beam_solver` — using your own code as a module
- `ax.fill_between()` with `where=` for colour-coded fills

**Structural concepts covered:**
- Shear force = sum of forces to the left of the section
- SFD rules: steps at point loads, slope under UDL, zero at free ends
- V = 0 marks the location of maximum bending moment

## Before the Next Lesson

In **Lesson 6** you'll integrate the SFD to get the bending moment diagram. The key insight: the slope of the BMD at any point equals the shear force at that point.